# Exemplo de uso de SQLite

## Bibliotecas

In [20]:
import sqlite3

## Conectando com Banco de Dados

In [21]:
conn = sqlite3.connect('loja.db') # Conectando ao banco de dados SQLite e criando uma conexao
cursor = conn.cursor() # Criando um cursor para executar comandos SQL

## Deletando Tabela Caso Exista

In [22]:
cursor.execute('DROP TABLE IF EXISTS pedido') # Executando um comando SQL para excluir a tabela 'produto' se ela existir

## Criando as Tabelas

In [23]:
cursor.execute('''

    CREATE TABLE pedido (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        cliente TEXT NOT NULL,
        data_pedido TEXT NOT NULL,
        status TEXT NOT NULL
    );
''')

conn.commit()

## Alterando as Tabelas

In [24]:
cursor.execute('''

    ALTER TABLE pedido ADD COLUMN forma_pagamento TEXT;

''')

conn.commit()

## Inserindo na Tabela

In [25]:
cursor.execute('''

    INSERT INTO pedido (cliente, data_pedido, status, forma_pagamento) VALUES
    ('João Silva', '2024-06-01', 'Pendente', 'Cartão de Crédito');


''')

conn.commit()

### Inserindo uma Lista

In [26]:
pedidos = [
    ('Maria Oliveira', '2024-06-02', 'Concluído', 'Boleto Bancário'),
    ('Carlos Pereira', '2024-06-03', 'Pendente', 'Pix')
]

cursor.executemany('''
                   
    INSERT INTO pedido (cliente, data_pedido, status, forma_pagamento) VALUES 
    (?, ?, ?, ?)
                   
''', pedidos)

conn.commit()

## Buscando Informacoes

In [27]:
cursor.execute('''
               
    SELECT * FROM pedido;

''')

for linha in cursor.fetchall():
    print(linha)

(1, 'João Silva', '2024-06-01', 'Pendente', 'Cartão de Crédito')
(2, 'Maria Oliveira', '2024-06-02', 'Concluído', 'Boleto Bancário')
(3, 'Carlos Pereira', '2024-06-03', 'Pendente', 'Pix')


# Exemplo de uso de SQL Alchemy -> RAW

## Bibliotecas

In [28]:
from sqlalchemy import create_engine, text

## Criando uma Engine

In [29]:
engine = create_engine('sqlite:///zoo.db')

## Conectando ao Banco e Executando Comando

In [30]:
with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS animais'))

## Criando Tabela

In [31]:
with engine.begin() as conn:
    conn.execute(text('''
    
        CREATE TABLE animais (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            especie TEXT NOT NULL,
            idade INTEGER NOT NULL
        );
    
    '''))

## Inserindo na Tabela

In [32]:
with engine.begin() as conn:
    conn.execute(text('''
    
        INSERT INTO animais (nome, especie, idade) VALUES
        ('Leão', 'Panthera leo', 5),
        ('Elefante', 'Loxodonta africana', 10),
        ('Girafa', 'Giraffa camelopardalis', 7);
    
    '''))

## Buscando Informacoes

In [33]:
with engine.begin() as conn:
    resultados = conn.execute(text('SELECT * FROM animais'))
    
for linha in resultados:
    print(linha)

(1, 'Leão', 'Panthera leo', 5)
(2, 'Elefante', 'Loxodonta africana', 10)
(3, 'Girafa', 'Giraffa camelopardalis', 7)


# Exemplo de uso de SQL Alchemy -> Expression Language

## Bibliotecas

In [34]:
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, select, insert

## Criando uma Engine e Conectando ao Banco

In [35]:
engine = create_engine('sqlite:///brasil.db')
metadata = MetaData()

## Criando Tabela

In [36]:
estados = Table('estados', metadata,
                Column('id', Integer, primary_key=True, autoincrement=True),
                Column('nome', String, nullable=False),
                Column('sigla', String, nullable=False)
)
estados.drop(engine, checkfirst=True)

In [37]:
metadata.create_all(engine)

## Inserindo e Buscando na Tabela

In [38]:
with engine.begin() as conn:
    conn.execute(insert(estados).values(nome='São Paulo', sigla='SP'))
    resultado = conn.execute(select(estados))

print(resultado.fetchall())

[(1, 'São Paulo', 'SP')]


# ORM - Object Relational Mapper

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker, relationship

In [41]:
Base = declarative_base()

In [42]:
class Pedido(Base):
    __tablename__ = 'pedidos'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    cliente = Column(String, nullable=False)
    
    produtos = relationship('Produto', back_populates='pedido')

In [43]:
class Produto(Base):
    __tablename__ = 'produtos'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    nome = Column(String, nullable=False)
    preco = Column(Integer, nullable=False)
    pedido_id = Column(Integer, ForeignKey('pedidos.id'))

    pedido = relationship('Pedido', back_populates='produtos')



In [44]:
engine = create_engine('sqlite:///vendas.db')

In [45]:
Base.metadata.create_all(engine)

In [46]:
Session = sessionmaker(bind=engine)
session = Session()

In [47]:
pedido1 = Pedido(cliente='Ana')
pedido2 = Pedido(cliente='Bruno')

produto1 = Produto(nome='Camiseta', preco=50, pedido=pedido1)
produto2 = Produto(nome='Calça', preco=100, pedido=pedido1)
produto3 = Produto(nome='Tênis', preco=150, pedido=pedido2)

session.add_all([pedido1, pedido2, produto1, produto2, produto3])
session.commit()

In [48]:
pedidos = session.query(Pedido).all()

for pedido in pedidos:
    print(f'Pedido ID: {pedido.id}, Cliente: {pedido.cliente}')
    for produto in pedido.produtos:
        print(f'  Produto: {produto.nome}, Preço: {produto.preco}')

Pedido ID: 1, Cliente: Ana
  Produto: Camiseta, Preço: 50
  Produto: Calça, Preço: 100
Pedido ID: 2, Cliente: Bruno
  Produto: Tênis, Preço: 150
